# 01 — Exploración de Datos (EDA)
**AgroVision AI** · Concurso Datos al Ecosistema 2026

## 1. Configuración

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({'figure.figsize': (12, 5), 'axes.spines.top': False,
                     'axes.spines.right': False, 'font.size': 11})
print("Librerías cargadas ✓")

## 2. Carga de datos EVA desde datos.gov.co

In [ ]:
import requests

URL = "https://www.datos.gov.co/resource/uejq-wxrr.json"
params = {
    "$limit": 5000,
    "$where": "rendimiento IS NOT NULL AND rendimiento > 0",
    "$order": "a_o DESC"
}
r = requests.get(URL, params=params, timeout=60)
df_raw = pd.DataFrame(r.json())
print(f"Registros descargados: {len(df_raw)}")
print(f"Columnas: {list(df_raw.columns)}")

## 3. Vista previa

In [ ]:
df_raw.head(10)

## 4. Información general del dataset

In [ ]:
print(f"Shape: {df_raw.shape}")
print(f"\nTipos de datos:\n{df_raw.dtypes}")
print(f"\nNulos por columna:\n{df_raw.isnull().sum().sort_values(ascending=False).head(10)}")

## 5. Renombrar columnas canónicas

In [ ]:
rename_map = {
    'a_o': 'anio', 'rea_sembrada': 'area_sembrada',
    'rea_cosechada': 'area_cosechada', 'producci_n': 'produccion',
}
df = df_raw.rename(columns={k:v for k,v in rename_map.items() if k in df_raw.columns})

for col in ['area_sembrada','area_cosechada','produccion','rendimiento','anio']:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

print("Columnas canónicas:", list(df.columns))

## 6. Distribución de registros por departamento

In [ ]:
top_dptos = df['departamento'].value_counts().head(15)
fig, ax = plt.subplots()
top_dptos.plot(kind='barh', ax=ax, color='#2D6A4F')
ax.set_xlabel('Registros')
ax.set_title('Top 15 departamentos por número de registros EVA')
plt.tight_layout()
plt.savefig('distribución_departamentos.png', dpi=150, bbox_inches='tight')
plt.show()
print(top_dptos)

## 7. Distribución de cultivos

In [ ]:
top_cultivos = df['cultivo'].value_counts().head(20)
fig, ax = plt.subplots(figsize=(12,6))
top_cultivos.plot(kind='bar', ax=ax, color='#52B788')
ax.set_xlabel('Cultivo')
ax.set_ylabel('Registros')
ax.set_title('Top 20 cultivos por número de registros EVA')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('distribución_cultivos.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Distribución del rendimiento (variable target)

In [ ]:
df_clean = df[df['rendimiento'] > 0].dropna(subset=['rendimiento'])
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df_clean['rendimiento'], bins=50, color='#1B4332', edgecolor='white')
axes[0].set_title('Distribución rendimiento (t/ha)')
axes[0].set_xlabel('Rendimiento (t/ha)')
axes[0].set_ylabel('Frecuencia')

df_filt = df_clean[df_clean['rendimiento'] < 50]
axes[1].hist(df_filt['rendimiento'], bins=50, color='#52B788', edgecolor='white')
axes[1].set_title('Rendimiento < 50 t/ha (sin outliers extremos)')
axes[1].set_xlabel('Rendimiento (t/ha)')
plt.tight_layout()
plt.savefig('distribución_rendimiento.png', dpi=150, bbox_inches='tight')
plt.show()
print(df_filt['rendimiento'].describe())

## 9. Cobertura temporal

In [ ]:
by_year = df.groupby('anio').size().reset_index(name='registros')
fig, ax = plt.subplots()
ax.bar(by_year['anio'].astype(str), by_year['registros'], color='#2D6A4F')
ax.set_title('Registros EVA por año')
ax.set_xlabel('Año')
ax.set_ylabel('Registros')
plt.tight_layout()
plt.savefig('cobertura_temporal.png', dpi=150, bbox_inches='tight')
plt.show()
print(by_year)

## 10. Análisis de nulos

In [ ]:
nulos = df.isnull().sum()
nulos_pct = (nulos / len(df) * 100).round(2)
resumen = pd.DataFrame({'nulos': nulos, 'porcentaje': nulos_pct})
resumen = resumen[resumen['nulos'] > 0].sort_values('porcentaje', ascending=False)
print("Columnas con valores nulos:")
print(resumen)

fig, ax = plt.subplots()
resumen['porcentaje'].plot(kind='bar', ax=ax, color='#F59E0B')
ax.set_title('Porcentaje de nulos por columna')
ax.set_ylabel('%')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('nulos_por_columna.png', dpi=150, bbox_inches='tight')
plt.show()

## 11. Conclusiones EDA

- El dataset EVA tiene buena cobertura nacional (32 departamentos)
- La variable `rendimiento` tiene valores atípicos extremos que se filtrarán (> 50 t/ha)
- Los registros de 2024 tienen más nulos en área y producción porque la campaña aún no cierra
- Los cultivos más representados son: arroz, maíz, papa, café y frijol